In [ ]:
from pinoco import ODEEquation, SimulatedTrajectoryDataset, SubtrajectoryDataset, SubtrajectoryView, PINNTrainDataset, TorchODEResidual, make_pinn_dataloader

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset

from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

from collections import defaultdict

from solis_nn import SOLIS, get_global_signal_stats, get_global_time_scale, MultitrajectoryIPINN, SimpleIPINN, save_model_package, load_checkpoint
from solis_training import train_epoch, train_epoch_ipinn, train_epoch_simple_ipinn
from solis_eval import eval_model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
print(f"Using device: {device}")

In [ ]:
# 1. Equation
# Duffing Oscillator Equation
eq_gt = ODEEquation(
    eqs=[
        "Eq(D(y,2), -delta*D(y,1) - alpha*y(t) - beta*y(t)**3 + u)"
    ],
    params={"delta": 0.2, "alpha": -1.0, "beta": 1.0, "gamma": 0.3, "omega": 1.2, },
    dependent_variables=["y"],
    exogenous_functions=["u"],
)

def make_exo_duffing(
    kind: str,
    amp=1.0,
    freqs=(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50),
    noise_seed=0,
):
    kind = kind.lower()

    if kind == "multitone":
        return {
            "u": lambda t: 1.0 * torch.cos(0.5 * t) + 0.7 * torch.sin(2.3 * t)
        }

    if kind == "step":
        return {
            "u": lambda t: torch.full_like(t, float(amp))
        }

    if kind == "multisine":
        gen = torch.Generator(device="cpu")
        gen.manual_seed(int(noise_seed))
        freqs_t = torch.tensor(list(freqs), dtype=torch.float32)
        phases = 2.0 * torch.pi * torch.rand((len(freqs_t),), generator=gen)
        A = float(amp) / max(1, len(freqs_t)) ** 0.5  # keep RMS sane

        def u(t):
            tt = t.reshape(-1)
            s = 0.0
            for fi, ph in zip(freqs_t, phases):
                s = s + torch.sin(2.0 * torch.pi * fi * tt + ph)
            return (A * s).reshape_as(t)

        return {"u": u}

    raise ValueError(f"Unknown kind: {kind}")


# 2. Initial Conditions
N_traj_multitone = 3
N_traj_step = 2
N_traj_multisine = 3

y0_set_multitone = (torch.rand(N_traj_multitone, 2) - 0.5) * 5.0
y0_set_multitone[:, 1] = (torch.rand(N_traj_multitone) - 0.5) * 4.0

y0_set_step = (torch.rand(N_traj_step, 2) - 0.5) * 5.0
y0_set_step[:, 1] = (torch.rand(N_traj_step) - 0.5) * 4.0

y0_set_multisine = (torch.rand(N_traj_multisine, 2) - 0.5) * 5.0
y0_set_multisine[:, 1] = (torch.rand(N_traj_multisine) - 0.5) * 4.0

Tf = 16.0
NT = 1024

COMMON_SIM_KW = dict(
    t0=0.0,
    tf=Tf,
    T=NT,
    backend="torch",
    method="rk4",
    device="cpu",
    dtype=torch.float32,
    keep_graph=False,
)

# 3. Simulation 
gt_traj_multitone = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_multitone,
    y0_set=y0_set_multitone,
    exogenous_torch=make_exo_duffing(kind="multitone"),
    **COMMON_SIM_KW,
)
subtraj_multitone = SubtrajectoryView(
    SubtrajectoryDataset(gt_traj_multitone, subseq_len=256)
)

gt_traj_step = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_step,
    y0_set=y0_set_step, 
    exogenous_torch=make_exo_duffing(kind="step"),
    **COMMON_SIM_KW,
)
subtraj_step = SubtrajectoryView(
    SubtrajectoryDataset(gt_traj_step, subseq_len=256)
)

gt_traj_multisine = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_multisine,
    y0_set=y0_set_multisine,
    exogenous_torch=make_exo_duffing(kind="multisine", amp=1.0, noise_seed=0),
    **COMMON_SIM_KW,
)
subtraj_multisine = SubtrajectoryView(
    SubtrajectoryDataset(gt_traj_multisine, subseq_len=256)
)

# 5. Training sets (one per scenario, then concat)
pinn_multitone = PINNTrainDataset(
    subtraj_multitone,
    collocation_frac=1.0,
    num_data_per_traj=30,
    Tf=Tf,
    disjoint_sets=False,
    importance_y_weight=5,
)

pinn_step = PINNTrainDataset(
    subtraj_step,
    collocation_frac=1.0,
    num_data_per_traj=30,
    Tf=Tf,
    disjoint_sets=False,
    importance_y_weight=5,
)

pinn_multisine = PINNTrainDataset(
    subtraj_multisine,
    collocation_frac=1.0,
    num_data_per_traj=30,
    Tf=Tf,
    disjoint_sets=False,
    importance_y_weight=5,
)

pinn_train_dataset = ConcatDataset([pinn_multitone, pinn_step, pinn_multisine])

In [ ]:
loader = make_pinn_dataloader(
    pinn_train_dataset,
    n_traj_samples=8,
    n_data_samples=NT,
    seed=123,
    shuffle_trajs=False
)

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

# Pick one trajectory and its sampled training set
dset = gt_traj_multitone[0]
pdset = pinn_train_dataset[0]

# Continuous ODE solution
ax.plot(dset["t"].cpu(), dset["y"][:,0].cpu(),
        label="y(t) Solution", color="C0", linewidth=2, alpha=0.6)

ax.plot(dset["t"].cpu(), dset["y"][:,1].cpu(),
        label="y(t) Solution", color="C0", linewidth=2, alpha=0.6)

ax.plot(dset["t"].cpu(), dset["exo"]["u"].cpu(),
        label="u(t)", color="C4", linewidth=2, alpha=0.6)

# Data points
ax.scatter(pdset["t_data"].cpu(), pdset["y_data"][:,0].cpu(),
           label="Data Points", color="C1", marker="o", edgecolors="k")

ax.scatter(pdset["t_data"].cpu(), pdset["y_data"][:,1].cpu(),
           label="Data Points", color="C1", marker="o", edgecolors="k")

# Initial condition
ax.scatter(pdset["t_col"][0].cpu(), pdset["y0"][0].cpu(),
           label="Initial Value", color="C2", marker="s", s=80, edgecolors="k")

# Collocation points (y=0 just for visualization)
ax.scatter(pdset["t_col"].cpu(),
           torch.zeros_like(pdset["t_col"]).cpu(),
           label="Collocation Points", color="C3", marker="x")

# Styling
ax.set_xlabel("Time $t$")
ax.set_ylabel("$y(t)$")
ax.set_title(f"Trajectory")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()

plt.tight_layout()
plt.show()

## Training

In [ ]:
global_t_min, global_t_scale = get_global_time_scale(pinn_train_dataset)
global_t_min = torch.tensor(global_t_min)
global_t_scale = torch.tensor(global_t_scale)

stats = get_global_signal_stats(pinn_train_dataset)

In [ ]:
# 2nd order hypothesis
plant_eq_y  = "Eq(D(y,1), v)"
plant_eq_dy = "Eq(D(v,1), -k*y - d*v + g*u)"

eq_pred = ODEEquation(
    eqs = [
        plant_eq_y,
        plant_eq_dy,
    ],  
    params={}, 
    dependent_variables=["y", "v"],
    exogenous_functions=["u", "k", "d", "g"],           
)

### solis Training

In [ ]:
cfg_solis = dict(
        x_dim=4, use_u_in_sol_net=True, use_intercept=False,
        include_abs_time=False, use_relative_time=True,
        use_moe=True, poly_order=3, ensure_positive_wn=False,
        include_abs=True, include_cross=True, include_energy=True,
        sol_net_layers=4, sol_net_hidden_dim=256, param_net_hidden_dim=256,
        use_fourier_time=True, include_u_in_params=True,
        traj_emb_dim=0, num_trajectories=len(pinn_train_dataset),
        use_input_normalization=True,
        context_hidden_dim=64, context_dim=32,
)

solis = SOLIS(**cfg_solis)

# solis.set_time_bounds(global_t_min.item(), (global_t_min + global_t_scale).item())
# solis.set_norm_stats(
#     y_mean=stats["y_mean"],
#     y_std=stats["y_std"],
#     v_mean=stats["v_mean"],
#     v_std=stats["v_std"],
#     u_mean=stats["u_mean"],
#     u_std=stats["u_std"],
# )

plant_params = list(solis.y_net.parameters())
if getattr(solis, "traj_emb", None) is not None:
    plant_params += list(solis.traj_emb.parameters())

if solis.use_moe:
    param_params = (
        list(solis.param_feat.parameters()) +
        list(solis.gating_net.parameters()) +
        [p for exp in solis.experts for p in exp.parameters()]
    )
else:
    param_params = list(solis.param_feat.parameters()) + list(solis.param_head.parameters())


use_velocity_data = True
use_velocity_ic   = True

if use_velocity_data:
    print("Using position and velocity data for training.")
    data_dim = 2
else:
    print("Using position-only data for training.")
    data_dim = 1

if use_velocity_ic:
    print("Using position and velocity in initial conditions.")
    ic_dim = 2
else:
    print("Using position-only in initial conditions.")
    ic_dim = 1

# Optimizers 
optimizer_plant  = torch.optim.AdamW(plant_params, lr=5e-4, weight_decay=5e-5)
optimizer_params = torch.optim.AdamW(param_params, lr=5e-4, weight_decay=5e-5)

# Losses and schedulers
criterion = nn.MSELoss()
residual = TorchODEResidual(eq_pred)
scheduler_plant  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_plant, mode="min", factor=0.5, patience=50)
scheduler_params = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_params, mode="min", factor=0.5, patience=50)

# Tensorboard
# now_str = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
# writer = SummaryWriter(log_dir=f"runs/solis-Phint-2nd{now_str}", flush_secs=3)
writer = None

In [ ]:
# ==========================================
# Configuration
# ==========================================
epochs = 5_000

# Define Phase Durations
phase1_epochs = 50  
phase2_epochs = 50  
epoch_period = phase1_epochs + phase2_epochs 

phint_cutoff_ratio = 0.1

# Track parameters
param_list = defaultdict(list)

# Progress bar
pbar = tqdm(range(1,epochs+1), desc="Initializing", ncols=120, dynamic_ncols=True)

prev_phase = 1

for epoch in pbar:
    phase_epoch = (epoch-1) % epoch_period

    # --------------------------------------
    # 1. Determine Phase & Hyperparameters
    # --------------------------------------
    if epoch < 51:
        current_phase = 0
        phase_desc = "Phase 0: Warmup"

        lambda_phy = 0.
        lambda_data = 10.
        lambda_ic = 100.

        current_phase = 1

    else:
        lambda_phy = 10.
        lambda_data = 10.
        lambda_ic = 100.

        if phase_epoch < phase1_epochs:
            current_phase = 1
            phase_desc = "Phase 1: Solution Learning"
            
        elif phase_epoch < (phase1_epochs + phase2_epochs):
            current_phase = 2
            phase_desc = "Phase 2: Parameter Identification"
    
    change_phase = not (current_phase == prev_phase)
    hint_decay = max(0.0, 1.0 - epoch / (epochs * phint_cutoff_ratio))
    
    # --------------------------------------
    # 2. Run Training Step
    # --------------------------------------
    step_info = train_epoch(
        model=solis,
        residual=residual,
        eq=eq_pred,
        loader=loader,
        ic_dim=ic_dim,
        data_dim=data_dim,
        optimizer_plant=optimizer_plant,
        optimizer_params=optimizer_params,
        criterion=criterion,
        phase=current_phase,  
        change_phase=change_phase,
        scheduler_plant=scheduler_plant,
        scheduler_params=scheduler_params,
        lambda_data=10.0,
        lambda_ic=100.0,
        lambda_phys=10.,        
        lambda_gate_reg=0.01,
        lambda_phy_hint=0, #.1 * hint_decay,
        lambda_tv=1,
        H_horizon=10,
        lambda_step=0.5,
        random_window="medium"
    )

    prev_phase = current_phase

    # --------------------------------------
    # 3. Logging
    # --------------------------------------
    # Log losses to TensorBoard
    if writer is not None:
        for key, value in step_info.items():
            writer.add_scalar(f"losses/{key}", value, epoch)
        writer.add_scalar("training/phase", current_phase, epoch)

    # Update tqdm status bar
    physics_loss = step_info['physics_loss']
    data_loss = step_info['data_loss']

    pbar.set_description(f"{phase_desc}")
    pbar.set_postfix({
        "Data Loss": f"{data_loss:.2e}",
        "Physics Loss" : f"{physics_loss:.2e}",
        "IC": f"{step_info['ic_loss']:.1e}"
    })

Save

In [ ]:
# --- Save them ---
save_dir = "./models/duffing"

# Assuming your model instances are named: model_solis, model_multi, model_simple
save_model_package(solis, cfg_solis, f"{save_dir}/solis.pth")

In [ ]:
load_dir = "./models/duffing"

solis_loaded = load_checkpoint(SOLIS, f"{load_dir}/solis.pth", device=device)

## Plots

### Training Set Solution & Parameter Field Learning

In [ ]:
import importlib
import solis_eval 
importlib.reload(solis_eval)
from solis_eval import eval_model, plot_surrogate_rollout, plot_phase_comparison

In [ ]:
eval_model({"SOLIS" : solis, 
            "SOLIS_loaded": solis_loaded,
            }, 
          ConcatDataset([subtraj_multisine, subtraj_step, subtraj_multitone]), 
          pinn_train_dataset, convert_params=False, plot_params=False,
          # save_path="./figures/duffing/duffing_solution.pdf",
          break_loop=True,
           )

### Surrogate Rollout
Test data

In [ ]:
# 2. Initial Conditions
N_traj_multitone = 1
N_traj_step = 1
N_traj_multisine = 1

y0_set_multitone_test = (torch.rand(N_traj_multitone, 2) - 0.5) * 5.0
y0_set_multitone_test[:, 1] = (torch.rand(N_traj_multitone) - 0.5) * 4.0

y0_set_step_test = (torch.rand(N_traj_step, 2) - 0.5) * 5.0
y0_set_step_test[:, 1] = (torch.rand(N_traj_step) - 0.5) * 4.0

y0_set_multisine_test = (torch.rand(N_traj_multisine, 2) - 0.5) * 5.0
y0_set_multisine_test[:, 1] = (torch.rand(N_traj_multisine) - 0.5) * 4.0

Tf = 16.0
NT = 1024

COMMON_SIM_KW = dict(
    t0=0.0,
    tf=Tf,
    T=NT,
    backend="torch",
    method="rk4",
    device="cpu",
    dtype=torch.float32,
    keep_graph=False,
)

test_traj_multitone = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_multitone,
    y0_set=y0_set_multitone_test,
    exogenous_torch=make_exo_duffing(kind="multitone"),
    **COMMON_SIM_KW,
)

test_traj_step = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_step,
    y0_set=y0_set_step_test,  
    exogenous_torch=make_exo_duffing(kind="step"),
    **COMMON_SIM_KW,
)

test_traj_multisine = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_multisine,
    y0_set=y0_set_multisine_test,
    exogenous_torch=make_exo_duffing(kind="multisine", amp=1.0, noise_seed=0),
    **COMMON_SIM_KW,
)

test_traj = ConcatDataset([test_traj_multitone, test_traj_step, test_traj_multisine])

In [ ]:
plot_surrogate_rollout({
    "SOLIS" : solis, 
    "SOLIS_loaded": solis_loaded,
    }, 
    eq_pred, 
    test_traj,
    0, convert_params=False, plot_params=False,
    use_running_average_v=False,use_running_average_y=True,blend_alpha=0.0,
    # save_path="./figures/duffing/duffing_surrogate.pdf"
    )

### State-space Vector Fields

In [ ]:
# Assuming your model and equations are ready
metrics = plot_phase_comparison(
    model=solis_loaded,
    eq_gt=eq_gt, 
    eq_pred=eq_pred,
    trajs=[traj["y"] for traj in ConcatDataset([subtraj_multisine, subtraj_step, subtraj_multitone])], 
    y_range=(-2.5, 2.5), 
    v_range=(-3, 3), 
    u0=0.0, # Plot autonomous dynamics (u=0)
    # save_path="figures/duffing/duffing_phase_solis.pdf"
)
print(metrics)

## Control with Surrogate

In [ ]:
from typing import Optional, Dict, Union, Callable

@torch.no_grad()
def simulate_closed_loop_solis_state_feedback(
    eq_gt,
    solis_model,
    *,
    x0: torch.Tensor,
    t0: float,
    dt: float,
    steps: int,
    # desired closed-loop design
    zeta_c: float = 0.9,
    omega_c: float = 6.0,
    # baseline safe gains (for blending)
    K0: torch.Tensor = None,  # shape (2,) -> [k1_0, k2_0]
    # safety / blending
    alpha: float = 1.0,        # constant blend factor in [0,1] (replace with confidence-based if you have it)
    g_min: float = 1e-3,       # lower bound on g to avoid division blow-ups
    # gain limits
    k1_min: float = -1e6,
    k1_max: float =  1e6,
    k2_min: float = -1e6,
    k2_max: float =  1e6,
    # gain rate limits (per second)
    dk1_max: float = 1e6,
    dk2_max: float = 1e6,
    # actuator limits
    u_min: float = -10.0,
    u_max: float =  10.0,
    # reference (optional)
    ref: Optional[Dict[str, Callable[[torch.Tensor], torch.Tensor]]] = None,
    # probing excitation (optional)
    probe: Optional[Callable[[float], float]] = None,
    probe_on: Optional[Callable[[torch.Tensor, torch.Tensor, torch.Tensor], bool]] = None,
    # integration
    method: str = "rk4",
    device: Optional[torch.device] = None,
    dtype: torch.dtype = torch.float32,
):
    """
    Closed-loop simulation with SOLIS-scheduled state feedback.

    Assumptions:
      - State is x = [y, v] (shape (2,) or (B,2)).
      - solis_model.predict_params(x) -> (k_hat, d_hat, g_hat) each broadcastable to (B,1) or (B,).
      - Control law (tracking form):
          e = y - r(t), de = v - rdot(t)
          u = u_ff + u_fb + delta_u
          u_fb = -k1*e - k2*de
        with pole-matching scheduled gains:
          k2_raw = (2*zeta_c*omega_c - d_hat) / g_hat_eff
          k1_raw = (omega_c^2 - k_hat) / g_hat_eff
        and safety: blend with baseline K0, saturate, rate-limit, saturate u.

    Returns:
      dict with trajectories: t, x, u, k_hat, d_hat, g_hat, k1, k2, r, rdot, rddot
    """
    if device is None:
        device = x0.device if isinstance(x0, torch.Tensor) else torch.device("cpu")

    # Ensure x is torch
    if not isinstance(x0, torch.Tensor):
        x = torch.tensor(x0, device=device, dtype=dtype)
    else:
        x = x0.to(device=device, dtype=dtype)

    # Make batched shape (B,2)
    if x.ndim == 1:
        x = x.unsqueeze(0)
    assert x.shape[-1] == 2, "Expected x = [y,v] with shape (B,2) or (2,)"

    B = x.shape[0]

    if K0 is None:
        # conservative baseline if not provided
        K0 = torch.tensor([0.0, 0.0], device=device, dtype=dtype)
    else:
        K0 = K0.to(device=device, dtype=dtype).reshape(2)

    # current scheduled gains (start at baseline)
    k1 = K0[0].expand(B).clone()
    k2 = K0[1].expand(B).clone()

    # Storage
    t_hist = torch.empty((steps + 1,), device=device, dtype=dtype)
    x_hist = torch.empty((steps + 1, B, 2), device=device, dtype=dtype)
    u_hist = torch.empty((steps, B), device=device, dtype=dtype)

    k_hat_hist = torch.empty((steps, B), device=device, dtype=dtype)
    d_hat_hist = torch.empty((steps, B), device=device, dtype=dtype)
    g_hat_hist = torch.empty((steps, B), device=device, dtype=dtype)
    k1_hist = torch.empty((steps, B), device=device, dtype=dtype)
    k2_hist = torch.empty((steps, B), device=device, dtype=dtype)

    r_hist = torch.empty((steps, B), device=device, dtype=dtype)
    rdot_hist = torch.empty((steps, B), device=device, dtype=dtype)
    rddot_hist = torch.empty((steps, B), device=device, dtype=dtype)

    # Reference functions (default: regulation to 0)
    if ref is None:
        ref = {}
    r_fun = ref.get("r", lambda tt: torch.zeros((B, 1), device=device, dtype=dtype))
    rd_fun = ref.get("rdot", lambda tt: torch.zeros((B, 1), device=device, dtype=dtype))
    rdd_fun = ref.get("rddot", lambda tt: torch.zeros((B, 1), device=device, dtype=dtype))

    t = torch.tensor(float(t0), device=device, dtype=dtype)
    dt_t = torch.tensor(float(dt), device=device, dtype=dtype)

    t_hist[0] = t
    x_hist[0] = x

    # Useful constants
    two_zeta_omega = 2.0 * float(zeta_c) * float(omega_c)
    omega2 = float(omega_c) ** 2

    u_step = torch.zeros((1,), device=device, dtype=dtype)  # initial control input

    # Main simulation loop
    for n in range(steps):
        # Time as (B,1) for reference and exogenous
        tt = t.expand(B, 1)

        # Reference signals
        r = r_fun(tt).reshape(B)
        rdot = rd_fun(tt).reshape(B)
        rddot = rdd_fun(tt).reshape(B)

        y = x[:, 0]
        v = x[:, 1]
        e = y - r
        de = v - rdot

        # 1) SOLIS parameter inference
        params,gate = solis_model.predict_params(x, u=u_step.unsqueeze(0), detach_state=True)  # expected shapes (B,) or (B,1)
        k_hat = params[:, 0]
        d_hat = params[:, 1]
        g_hat = params[:, 2]

        k_hat = k_hat.reshape(B)
        d_hat = d_hat.reshape(B)
        g_hat = g_hat.reshape(B)

        # 2) Safety: clamp g away from zero
        g_eff = torch.clamp(g_hat, min=g_min)

        # 3) Scheduled gains by pole matching
        k2_raw = (two_zeta_omega - d_hat) / g_eff
        k1_raw = (omega2 - k_hat) / g_eff

        # 4) Blend with baseline safe gains
        a = float(alpha)
        k1_des = (1.0 - a) * K0[0] + a * k1_raw
        k2_des = (1.0 - a) * K0[1] + a * k2_raw

        # 5) Saturate gains
        k1_des = torch.clamp(k1_des, k1_min, k1_max)
        k2_des = torch.clamp(k2_des, k2_min, k2_max)

        # 6) Rate-limit gains
        dk1 = torch.clamp(k1_des - k1, -dk1_max * dt, dk1_max * dt)
        dk2 = torch.clamp(k2_des - k2, -dk2_max * dt, dk2_max * dt)
        k1 = k1 + dk1
        k2 = k2 + dk2

        # 7) Control evaluation
        u_fb = -(k1 * e + k2 * de)

        # Optional feedforward: u_ff = rddot / g_eff
        u_ff = rddot / g_eff

        # Optional probing excitation
        delta_u = torch.zeros_like(u_fb)
        if probe is not None:
            du = float(probe(float(t.item())))
            if probe_on is None:
                # Always on if provided (you can gate this externally)
                delta_u = delta_u + du
            else:
                # probe_on takes (x, u_fb, params_scalar) and returns bool; customize as needed
                do_probe = probe_on(x, u_fb, g_hat)
                if do_probe:
                    delta_u = delta_u + du

        u_raw = u_ff + u_fb + delta_u
        u = torch.clamp(u_raw, u_min, u_max)

        # 8) Step the ground-truth plant
        u_step = u.clone()  # (B,)
        exogenous = {"u": (lambda _tt, uu=u_step: uu.view(B, 1))}

        x_next = eq_gt.simulate_one_step(
            t=t, x=x, dt=dt_t, exogenous=exogenous, params_override=None, method=method
        )

        # Store
        t_hist[n + 1] = t + dt_t
        x_hist[n + 1] = x_next

        u_hist[n] = u
        k_hat_hist[n] = k_hat
        d_hat_hist[n] = d_hat
        g_hat_hist[n] = g_hat
        k1_hist[n] = k1
        k2_hist[n] = k2

        r_hist[n] = r
        rdot_hist[n] = rdot
        rddot_hist[n] = rddot

        # advance
        x = x_next
        t = t + dt_t

    return {
        "t": t_hist.detach().cpu(),
        "x": x_hist.detach().cpu(),      # (T+1,B,2)
        "u": u_hist.detach().cpu(),      # (T,B)
        "k_hat": k_hat_hist.detach().cpu(),
        "d_hat": d_hat_hist.detach().cpu(),
        "g_hat": g_hat_hist.detach().cpu(),
        "k1": k1_hist.detach().cpu(),
        "k2": k2_hist.detach().cpu(),
        "r": r_hist.detach().cpu(),
        "rdot": rdot_hist.detach().cpu(),
        "rddot": rddot_hist.detach().cpu(),
    }

In [ ]:
x0 = torch.tensor([1, 0.5])  # initial [y,v]
out = simulate_closed_loop_solis_state_feedback(
    eq_gt, solis_loaded,
    x0=x0, t0=0.0, dt=1e-3, steps=10_000,
    zeta_c=0.9, omega_c=6.0,
    K0=torch.tensor([2.0, 1.5]),
    alpha=0.7,
    g_min=1e-2,
    u_min=-5.0, u_max=5.0,
    ref={"r": lambda tt: 0.2*torch.sin(1.0*tt), "rdot": lambda tt: 0.2*1.0*torch.cos(1.0*tt),
         "rddot": lambda tt: -0.2*1.0*1.0*torch.sin(1.0*tt)},
    method="rk4",
)

In [ ]:
plt.figure()
plt.plot(out["t"], out["x"][:,0,0], label="y (position)")
plt.plot(out["t"], out["x"][:,0,1], label="v (velocity)")
# plt.plot(out["t"][:-1], out["u"][:,0], label="u (control input)", linestyle='--')
plt.plot(out["t"][:-1], out["r"][:,0], label="r (reference)", linestyle=':')
plt.plot(out["t"][:-1], out["rdot"][:,0], label="rdot (reference)", linestyle=':')
# plt.plot(out["t"][:-1], out["k1"][:,0], label="Gain k1", linestyle=':')
# plt.plot(out["t"][:-1], out["k2"][:,0], label="Gain k2", linestyle=':')
plt.xlabel("Time (s)")
plt.ylabel("Position / Velocity")
plt.legend()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ----------------------------
# Global plotting style
# ----------------------------
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 2.0,
})

# ----------------------------
# Reference generators
# ----------------------------
def make_sine_ref(A=0.2, w=1.0):
    return {
        "r":     lambda tt: A * torch.sin(w * tt),
        "rdot":  lambda tt: A * w * torch.cos(w * tt),
        "rddot": lambda tt: -A * (w ** 2) * torch.sin(w * tt),
    }

def make_step_ref(level=0.5, t_step=1.0):
    def r(tt):
        return level * (tt >= t_step).float()
    def rdot(tt):
        return torch.zeros_like(tt)
    def rddot(tt):
        return torch.zeros_like(tt)
    return {"r": r, "rdot": rdot, "rddot": rddot}

def make_chirp_ref(A=0.15, w0=0.5, rate=0.12):
    # phase(t) = w0*t + 0.5*rate*t^2
    def r(tt):
        phi = w0 * tt + 0.5 * rate * tt**2
        return A * torch.sin(phi)
    def rdot(tt):
        phi = w0 * tt + 0.5 * rate * tt**2
        dphi = w0 + rate * tt
        return A * dphi * torch.cos(phi)
    def rddot(tt):
        phi = w0 * tt + 0.5 * rate * tt**2
        dphi = w0 + rate * tt
        ddphi = rate
        return A * (ddphi * torch.cos(phi) - dphi**2 * torch.sin(phi))
    return {"r": r, "rdot": rdot, "rddot": rddot}

# ----------------------------
# Simulation helper
# ----------------------------
def run_case(
    eq_gt,
    solis_model,
    x0,
    ref,
    *,
    t0=0.0,
    dt=1e-3,
    steps=10000,
    zeta_c=0.9,
    omega_c=6.0,
    K0=(2.0, 1.5),
    alpha=0.7,
    g_min=1e-2,
    u_min=-5.0,
    u_max=5.0,
    method="rk4",
):
    out = simulate_closed_loop_solis_state_feedback(
        eq_gt,
        solis_model,
        x0=torch.tensor(x0, dtype=torch.float32),
        t0=t0,
        dt=dt,
        steps=steps,
        zeta_c=zeta_c,
        omega_c=omega_c,
        K0=torch.tensor(K0, dtype=torch.float32),
        alpha=alpha,
        g_min=g_min,
        u_min=u_min,
        u_max=u_max,
        ref=ref,
        method=method,
    )
    return out

# ----------------------------
# Representative single-case figure
# ----------------------------
def plot_single_case(out, title=None, savepath=None):
    t = out["t"].numpy()
    y = out["x"][:, 0, 0].numpy()
    v = out["x"][:, 0, 1].numpy()

    tu = out["t"][:-1].numpy()
    u = out["u"][:, 0].numpy()
    r = out["r"][:, 0].numpy()
    k1 = out["k1"][:, 0].numpy()
    k2 = out["k2"][:, 0].numpy()

    fig, axs = plt.subplots(
        3, 1, figsize=(8.0, 7.0), sharex=True,
        gridspec_kw={"height_ratios": [2.2, 1.2, 1.2]}
    )

    # Tracking panel
    axs[0].plot(t, y, color="#1f77b4", label="Output $y(t)$")
    axs[0].plot(tu, r, color="#d62728", linestyle="--", label="Reference $r(t)$")
    axs[0].set_ylabel("Output")
    axs[0].grid(True, alpha=0.25)
    axs[0].legend(loc="upper right", frameon=False)

    # Control panel
    axs[1].plot(tu, u, color="#2ca02c", label="Control $u(t)$")
    axs[1].set_ylabel("Control")
    axs[1].grid(True, alpha=0.25)
    axs[1].legend(loc="upper right", frameon=False)

    # Gains panel
    axs[2].plot(tu, k1, color="#9467bd", label="$k_1(t)$")
    axs[2].plot(tu, k2, color="#ff7f0e", label="$k_2(t)$")
    axs[2].set_ylabel("Gains")
    axs[2].set_xlabel("Time (s)")
    axs[2].grid(True, alpha=0.25)
    axs[2].legend(loc="upper right", ncol=2, frameon=False)

    if title is not None:
        fig.suptitle(title, y=0.98)

    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, bbox_inches="tight")
    plt.show()

# ----------------------------
# Multi-case figure: same reference, different initial conditions
# ----------------------------
def plot_multiple_initial_conditions(
    eq_gt,
    solis_model,
    x0_list,
    ref,
    *,
    labels=None,
    t0=0.0,
    dt=1e-3,
    steps=10000,
    zeta_c=0.9,
    omega_c=6.0,
    K0=(2.0, 1.5),
    alpha=0.7,
    g_min=1e-2,
    u_min=-5.0,
    u_max=5.0,
    method="rk4",
    savepath=None,
):
    if labels is None:
        labels = [f"IC {i+1}" for i in range(len(x0_list))]

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b"]

    fig, ax = plt.subplots(figsize=(8.0, 4.6))

    ref_drawn = False
    for i, (x0, label) in enumerate(zip(x0_list, labels)):
        out = run_case(
            eq_gt, solis_model, x0, ref,
            t0=t0, dt=dt, steps=steps,
            zeta_c=zeta_c, omega_c=omega_c,
            K0=K0, alpha=alpha, g_min=g_min,
            u_min=u_min, u_max=u_max, method=method
        )

        t = out["t"].numpy()
        y = out["x"][:, 0, 0].numpy()
        tu = out["t"][:-1].numpy()
        r = out["r"][:, 0].numpy()

        ax.plot(t, y, color=colors[i % len(colors)], label=label)

        if not ref_drawn:
            ax.plot(tu, r, color="black", linestyle="--", linewidth=2.2, label="Reference")
            ref_drawn = True

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Output")
    ax.set_title("Tracking under different initial conditions")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, bbox_inches="tight")
    plt.show()

# ----------------------------
# Multi-case figure: different references
# ----------------------------
def plot_different_references(
    eq_gt,
    solis_model,
    x0,
    ref_dict,
    *,
    t0=0.0,
    dt=1e-3,
    steps=10000,
    zeta_c=0.9,
    omega_c=6.0,
    K0=(2.0, 1.5),
    alpha=0.7,
    g_min=1e-2,
    u_min=-5.0,
    u_max=5.0,
    method="rk4",
    savepath=None,
):
    n = len(ref_dict)

    fig, axs = plt.subplots(
        n, 1,
        figsize=(3.0, 3.35),   # good for ~900x1000 at 300 dpi
        sharex=True
    )

    if n == 1:
        axs = [axs]

    title_map = {
        "Sinusoidal reference": "Sinusoid",
        "Step reference": "Step",
        "Chirp reference": "Chirp",
    }

    for ax, (name, ref) in zip(axs, ref_dict.items()):
        out = run_case(
            eq_gt, solis_model, x0, ref,
            t0=t0, dt=dt, steps=steps,
            zeta_c=zeta_c, omega_c=omega_c,
            K0=K0, alpha=alpha, g_min=g_min,
            u_min=u_min, u_max=u_max, method=method
        )

        t = out["t"].numpy()
        y = out["x"][:, 0, 0].numpy()
        tu = out["t"][:-1].numpy()
        r = out["r"][:, 0].numpy()

        ax.plot(t, y, color="#1f77b4", lw=2.0)
        ax.plot(tu, r, color="#d62728", linestyle="--", lw=2.0)

        ax.set_ylabel("Output", fontsize=10)
        ax.set_title(title_map.get(name, name), fontsize=11, pad=4)
        ax.grid(True, alpha=0.2)
        ax.tick_params(axis="both", labelsize=9)

    axs[-1].set_xlabel("Time (s)", fontsize=11)

    # Shared legend for the whole figure
    handles = [
        Line2D([0], [0], color="#1f77b4", lw=2.0, label=r"Output $y(t)$"),
        Line2D([0], [0], color="#d62728", lw=2.0, linestyle="--", label=r"Reference $r(t)$"),
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=2,
        frameon=False,
        fontsize=9,
        bbox_to_anchor=(0.5, 0.995)
    )

    fig.tight_layout(rect=[0, 0, 1, 0.94])

    if savepath is not None:
        fig.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.show()

# Different references from the same initial condition
ref_cases = {
    "Sinusoidal reference": make_sine_ref(A=0.2, w=1.0),
    "Step reference": make_step_ref(level=0.35, t_step=1.0),
    "Chirp reference": make_chirp_ref(A=0.15, w0=0.5, rate=0.12),
}

plot_different_references(
    eq_gt,
    solis_loaded,
    x0=[1.0, 0.5],
    ref_dict=ref_cases,
    t0=0.0,
    dt=1e-3,
    steps=10000,
    zeta_c=0.9,
    omega_c=6.0,
    K0=(2.0, 1.5),
    alpha=0.7,
    g_min=1e-2,
    u_min=-5.0,
    u_max=5.0,
    method="rk4",
    savepath="solstice_tracking_reference_comparison.png",
)